# Control-Operation Evaluation

This notebook evaluates three locked FYS control plans on the same cases and seeds:

1. **Oracle FYS control**: original FYS timing with an oracle Stage 3 mask.
2. **Oracle Stage 2 edit-logit gate**: the same pipeline plus oracle spatial gating of edit-token logits in Stage 2.
3. **Part-to-edit logit transfer**: the same pipeline plus dynamic transfer of part-token spatial logits to edit-token logits in Stage 2.

The notebook does not run FLUX and does not modify inference artifacts. It validates completed runs, computes image-quality and preservation metrics, creates a manual-review table, and renders all cases for direct comparison.

## 1. Setup

Run this notebook from any working directory. STRICT_INPUTS requires every method to contain the same case/seed runs before evaluation proceeds. LPIPS is optional; when unavailable, existing non-empty LPIPS values are preserved.

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import Markdown, display

try:
    import torch
    import lpips
    HAS_LPIPS = True
except Exception:
    torch = None
    lpips = None
    HAS_LPIPS = False

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not locate repository root")
    REPO_ROOT = REPO_ROOT.parent

MANIFEST_PATH = REPO_ROOT / "core/data/partedit_subset/pilot_12_manifest.json"
CONTROL_ROOT = REPO_ROOT / "core/results/control_operations"
EVAL_ROOT = REPO_ROOT / "core/results/control_operations_eval"
FIGURE_ROOT = EVAL_ROOT / "figures"
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

METHODS = {
    "oracle_fys": "oracle_fys_control",
    "stage2_edit_gate": "oracle_stage2_edit_logit_gate",
    "part_to_edit": "part_to_edit_logit_transfer",
}
STRICT_INPUTS = True
EXPECTED_CASES = 12

print("repo:", REPO_ROOT)
print("LPIPS available:", HAS_LPIPS)

## 2. Audit completed runs

A valid run must contain the generated image, immutable run configuration, resolved control plan, log, and control trace. This prevents partial jobs from silently entering the comparison.

In [ ]:
manifest = pd.DataFrame(json.loads(MANIFEST_PATH.read_text(encoding="utf-8")))
if len(manifest) != EXPECTED_CASES or manifest["case_uid"].nunique() != EXPECTED_CASES:
    raise ValueError(f"Expected {EXPECTED_CASES} unique cases, found {len(manifest)} rows")

records = []
for method, plan_name in METHODS.items():
    method_root = CONTROL_ROOT / plan_name
    for case in manifest.to_dict("records"):
        case_root = method_root / case["case_uid"]
        for seed_dir in sorted(case_root.glob("seed_*")):
            image_candidates = sorted(seed_dir.glob("img_*.jpg")) + sorted(seed_dir.glob("img_*.png"))
            required = {
                "run_config": seed_dir / "run_config.json",
                "resolved_plan": seed_dir / "resolved_control_plan.json",
                "run_log": seed_dir / "run.log",
                "control_trace": seed_dir / "tdm/control_trace.json",
            }
            missing = [name for name, path in required.items() if not path.exists()]
            if not image_candidates:
                missing.append("generated_image")
            seed = int(seed_dir.name.split("_")[-1])
            records.append({
                **case,
                "method": method,
                "plan_name": plan_name,
                "seed": seed,
                "run_uid": f"{case['case_uid']}_seed_{seed:03d}",
                "run_dir": seed_dir.relative_to(REPO_ROOT).as_posix(),
                "edited_image": image_candidates[0].relative_to(REPO_ROOT).as_posix() if image_candidates else None,
                "missing_artifacts": ", ".join(missing),
                "complete": not missing,
            })

runs = pd.DataFrame(records)
if runs.empty:
    raise FileNotFoundError(
        f"No control-operation outputs found under {CONTROL_ROOT}. Run the three plans with --execute first."
    )

audit = runs.groupby("method", as_index=False).agg(
    runs=("run_uid", "size"),
    cases=("case_uid", "nunique"),
    complete=("complete", "sum"),
)
display(audit)

seed_sets = runs.groupby(["method", "case_uid"])["seed"].apply(lambda value: tuple(sorted(value))).reset_index()
print("seed sets by method:", seed_sets.groupby("method")["seed"].apply(lambda value: sorted(set(value))).to_dict())

if STRICT_INPUTS:
    incomplete = runs.loc[~runs["complete"], ["method", "run_uid", "missing_artifacts"]]
    if not incomplete.empty:
        raise FileNotFoundError("Incomplete runs:\n" + incomplete.to_string(index=False))
    expected = set(manifest["case_uid"])
    case_sets = {method: set(group["case_uid"]) for method, group in runs.groupby("method")}
    if set(case_sets) != set(METHODS) or any(cases != expected for cases in case_sets.values()):
        raise ValueError("Methods do not contain the same 12-case manifest")
    method_seed_sets = {
        method: set(zip(group["case_uid"], group["seed"]))
        for method, group in runs.groupby("method")
    }
    if len({frozenset(value) for value in method_seed_sets.values()}) != 1:
        raise ValueError("Methods do not contain identical case/seed combinations")

## 3. Metric definitions

Automatic metrics are separated by purpose:

- **Outside-mask L1 / PSNR / SSIM / LPIPS** measure non-target preservation.
- **Inside-mask L1** measures how strongly the requested region changed, but does not establish semantic success.
- **Human local-edit success** is required to judge whether the requested edit actually occurred.
- Repeated seeds with identical image hashes are reported but not treated as independent outputs.

In [ ]:
def load_rgb(path: Path, size: tuple[int, int] | None = None) -> np.ndarray:
    image = Image.open(path).convert("RGB")
    if size is not None and image.size != size:
        image = image.resize(size, Image.Resampling.BICUBIC)
    return np.asarray(image, dtype=np.float32) / 255.0


def load_mask(path: Path, size: tuple[int, int]) -> np.ndarray:
    image = Image.open(path).convert("L").resize(size, Image.Resampling.NEAREST)
    return np.asarray(image) > 0


def masked_mean(values: np.ndarray, mask: np.ndarray) -> float:
    if not np.any(mask):
        return float("nan")
    if values.ndim == 3:
        return float(values[mask].mean())
    return float(values[mask].mean())


def masked_mse(source: np.ndarray, edited: np.ndarray, mask: np.ndarray) -> float:
    return masked_mean((source - edited) ** 2, mask)


def psnr_from_mse(mse: float) -> float:
    return float("inf") if mse == 0 else float(10 * np.log10(1.0 / mse))


def masked_global_ssim(source: np.ndarray, edited: np.ndarray, mask: np.ndarray) -> float:
    if not np.any(mask):
        return float("nan")
    source_values = source[mask].astype(np.float64)
    edited_values = edited[mask].astype(np.float64)
    source_mean = source_values.mean(axis=0)
    edited_mean = edited_values.mean(axis=0)
    source_var = source_values.var(axis=0)
    edited_var = edited_values.var(axis=0)
    covariance = ((source_values - source_mean) * (edited_values - edited_mean)).mean(axis=0)
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    score = (
        (2 * source_mean * edited_mean + c1) * (2 * covariance + c2)
        / ((source_mean ** 2 + edited_mean ** 2 + c1) * (source_var + edited_var + c2))
    )
    return float(np.mean(score))


def image_sha256(path: Path) -> str:
    pixels = np.asarray(Image.open(path).convert("RGB"), dtype=np.uint8)
    digest = hashlib.sha256()
    digest.update(np.asarray(pixels.shape, dtype=np.int64).tobytes())
    digest.update(pixels.tobytes())
    return digest.hexdigest()


LPIPS_MODEL = None
if HAS_LPIPS:
    LPIPS_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    LPIPS_MODEL = lpips.LPIPS(net="alex").to(LPIPS_DEVICE).eval()
    print("LPIPS device:", LPIPS_DEVICE)


def outside_lpips(source: np.ndarray, edited: np.ndarray, gt_mask: np.ndarray) -> float:
    if LPIPS_MODEL is None:
        return float("nan")
    edited_eval = edited.copy()
    edited_eval[gt_mask] = source[gt_mask]
    source_tensor = torch.from_numpy(source.transpose(2, 0, 1)).unsqueeze(0).float() * 2 - 1
    edited_tensor = torch.from_numpy(edited_eval.transpose(2, 0, 1)).unsqueeze(0).float() * 2 - 1
    with torch.no_grad():
        return float(LPIPS_MODEL(source_tensor.to(LPIPS_DEVICE), edited_tensor.to(LPIPS_DEVICE)).item())

## 4. Compute per-run metrics

The save policy is non-destructive: if LPIPS cannot be computed in the current environment, previously saved LPIPS values are retained by method and run UID.

In [ ]:
METRICS_PATH = EVAL_ROOT / "control_operation_metrics.csv"
existing_lpips = pd.DataFrame()
if METRICS_PATH.exists():
    existing = pd.read_csv(METRICS_PATH)
    if "outside_mask_lpips" in existing:
        existing_lpips = existing[["method", "run_uid", "outside_mask_lpips"]].copy()

metric_rows = []
for row in runs.loc[runs["complete"]].to_dict("records"):
    source_path = REPO_ROOT / row["source_image"]
    edited_path = REPO_ROOT / row["edited_image"]
    source_pil = Image.open(source_path).convert("RGB")
    image_size = source_pil.size
    source = np.asarray(source_pil, dtype=np.float32) / 255.0
    edited = load_rgb(edited_path, size=image_size)
    gt_mask = load_mask(REPO_ROOT / row["gt_mask"], image_size)
    outside = ~gt_mask
    abs_diff = np.abs(source - edited)
    outside_mse = masked_mse(source, edited, outside)

    metric_rows.append({
        "method": row["method"],
        "plan_name": row["plan_name"],
        "run_uid": row["run_uid"],
        "case_uid": row["case_uid"],
        "seed": row["seed"],
        "part_size": row["part_size"],
        "part": row["part"],
        "edit": row["edit"],
        "target_prompt": row["target_prompt"],
        "inside_mask_l1": masked_mean(abs_diff, gt_mask),
        "outside_mask_l1": masked_mean(abs_diff, outside),
        "outside_mask_psnr": psnr_from_mse(outside_mse),
        "outside_mask_ssim": masked_global_ssim(source, edited, outside),
        "outside_mask_lpips": outside_lpips(source, edited, gt_mask),
        "image_sha256": image_sha256(edited_path),
        "source_image": row["source_image"],
        "gt_mask": row["gt_mask"],
        "edited_image": row["edited_image"],
        "run_dir": row["run_dir"],
    })

metrics = pd.DataFrame(metric_rows)
if not existing_lpips.empty:
    metrics = metrics.merge(
        existing_lpips.rename(columns={"outside_mask_lpips": "outside_mask_lpips_existing"}),
        on=["method", "run_uid"],
        how="left",
    )
    preserve = metrics["outside_mask_lpips"].isna() & metrics["outside_mask_lpips_existing"].notna()
    metrics.loc[preserve, "outside_mask_lpips"] = metrics.loc[preserve, "outside_mask_lpips_existing"]
    metrics = metrics.drop(columns="outside_mask_lpips_existing")
    print("preserved LPIPS values:", int(preserve.sum()))

metrics.to_csv(METRICS_PATH, index=False)
print("saved:", METRICS_PATH)
print("LPIPS non-empty:", int(metrics["outside_mask_lpips"].notna().sum()))
display(metrics.head())

## 5. Aggregate without inflating identical seeds

The table first reports command runs and unique image hashes. Method summaries use one row per method, case, and image hash, so deterministic duplicate seeds do not create artificial confidence.

In [ ]:
uniqueness = metrics.groupby("method", as_index=False).agg(
    command_runs=("run_uid", "size"),
    unique_cases=("case_uid", "nunique"),
    unique_images=("image_sha256", "nunique"),
)
display(uniqueness)

unique_metrics = metrics.drop_duplicates(["method", "case_uid", "image_sha256"]).copy()
summary_metrics = [
    "inside_mask_l1",
    "outside_mask_l1",
    "outside_mask_psnr",
    "outside_mask_ssim",
    "outside_mask_lpips",
]
summary = unique_metrics.groupby("method")[summary_metrics].agg(["mean", "std", "median"])
SUMMARY_PATH = EVAL_ROOT / "control_operation_summary.csv"
summary.to_csv(SUMMARY_PATH)
display(summary.round(4))

case_metrics = unique_metrics.groupby(["method", "case_uid"], as_index=False)[summary_metrics].mean()
baseline = case_metrics.loc[
    case_metrics["method"] == "oracle_fys", ["case_uid", *summary_metrics]
]
pairwise_rows = []
for method in ["stage2_edit_gate", "part_to_edit"]:
    current = case_metrics.loc[case_metrics["method"] == method, ["case_uid", *summary_metrics]]
    paired = current.merge(
        baseline,
        on="case_uid",
        suffixes=("_method", "_baseline"),
        validate="one_to_one",
    )
    record = {"method": method, "paired_cases": len(paired)}
    for metric in summary_metrics:
        record[f"delta_{metric}"] = float(
            (paired[f"{metric}_method"] - paired[f"{metric}_baseline"]).mean()
        )
    pairwise_rows.append(record)

pairwise = pd.DataFrame(pairwise_rows)
PAIRWISE_PATH = EVAL_ROOT / "paired_metric_deltas_vs_oracle_fys.csv"
pairwise.to_csv(PAIRWISE_PATH, index=False)
display(pairwise.round(4))

seed_variability = (
    metrics.groupby(["method", "case_uid"])[summary_metrics]
    .std(ddof=0)
    .groupby("method")
    .mean()
)
display(Markdown("### Mean within-case variation across command seeds"))
display(seed_variability.round(6))

## 6. Full qualitative comparison

One seed is shown per case to avoid repeated visual rows. The red overlay is the GT target part. The generated columns compare the three control operations directly.

In [ ]:
def overlay_mask(image: np.ndarray, mask: np.ndarray, color=(1.0, 0.1, 0.1), alpha=0.45) -> np.ndarray:
    result = image.copy()
    color_array = np.asarray(color, dtype=np.float32)
    result[mask] = (1 - alpha) * result[mask] + alpha * color_array
    return np.clip(result, 0, 1)


common_seeds = sorted(set.intersection(*[
    set(group["seed"]) for _, group in metrics.groupby("method")
]))
DISPLAY_SEED = common_seeds[0]

figure, axes = plt.subplots(EXPECTED_CASES, 5, figsize=(18, 3.15 * EXPECTED_CASES))
column_titles = ["source", "GT part", "oracle FYS", "Stage 2 edit gate", "part -> edit"]
for col, title in enumerate(column_titles):
    axes[0, col].set_title(title, fontsize=12)

for row_index, case in enumerate(manifest.to_dict("records")):
    source_path = REPO_ROOT / case["source_image"]
    source_pil = Image.open(source_path).convert("RGB")
    source = np.asarray(source_pil, dtype=np.float32) / 255.0
    gt = load_mask(REPO_ROOT / case["gt_mask"], source_pil.size)
    images = [source, overlay_mask(source, gt)]
    for method in ["oracle_fys", "stage2_edit_gate", "part_to_edit"]:
        match = metrics[
            (metrics["method"] == method)
            & (metrics["case_uid"] == case["case_uid"])
            & (metrics["seed"] == DISPLAY_SEED)
        ]
        if len(match) != 1:
            raise ValueError(f"Expected one {method} row for {case['case_uid']} seed {DISPLAY_SEED}")
        images.append(load_rgb(REPO_ROOT / match.iloc[0]["edited_image"], size=source_pil.size))

    for col, image in enumerate(images):
        axes[row_index, col].imshow(image)
        axes[row_index, col].axis("off")
    axes[row_index, 0].set_ylabel(
        f"{case['case_uid']}\n{case['part']} -> {case['edit']}\n{case['part_size']}",
        rotation=0,
        ha="right",
        va="center",
        fontsize=9,
        labelpad=70,
    )

figure.suptitle(f"Control-operation comparison, seed {DISPLAY_SEED}", fontsize=15, y=0.998)
figure.tight_layout()
FULL_FIGURE_PATH = FIGURE_ROOT / f"control_operation_all_cases_seed_{DISPLAY_SEED:03d}.jpg"
figure.savefig(FULL_FIGURE_PATH, dpi=180, bbox_inches="tight", pil_kwargs={"quality": 95})
plt.show()
print("saved:", FULL_FIGURE_PATH)

## 7. Manual semantic assessment

Automatic pixel metrics cannot determine whether an alien head, rust, mesh, or another requested semantic edit succeeded. Score each unique case from the rendered comparison:

- local_edit_success_0_2: 0 absent/wrong, 1 partial or ambiguous, 2 clearly successful.
- non_target_preservation_0_2: 0 major unintended change, 1 noticeable drift, 2 well preserved.
- overall_visual_quality_0_2: 0 visibly broken, 1 acceptable with artifacts, 2 coherent and high quality.

Existing scores are merged back and never cleared.

In [ ]:
MANUAL_PATH = EVAL_ROOT / "manual_control_operation_review.csv"
manual_base = metrics.loc[metrics["seed"] == DISPLAY_SEED, [
    "method", "case_uid", "part_size", "part", "edit", "target_prompt",
    "source_image", "gt_mask", "edited_image",
]].sort_values(["case_uid", "method"]).reset_index(drop=True)
if len(manual_base) != EXPECTED_CASES * len(METHODS):
    raise ValueError("Manual review must contain exactly one displayed output per method and case")

score_columns = [
    "local_edit_success_0_2",
    "non_target_preservation_0_2",
    "overall_visual_quality_0_2",
    "short_note",
]
if MANUAL_PATH.exists():
    saved_manual = pd.read_csv(MANUAL_PATH)
    preserved_columns = [column for column in score_columns if column in saved_manual]
    manual_review = manual_base.merge(
        saved_manual[["method", "case_uid", *preserved_columns]],
        on=["method", "case_uid"],
        how="left",
    )
else:
    manual_review = manual_base.copy()

for column in score_columns:
    if column not in manual_review:
        manual_review[column] = ""
    manual_review[column] = manual_review[column].fillna("")

manual_review.to_csv(MANUAL_PATH, index=False)
print("saved without clearing existing scores:", MANUAL_PATH)
display(manual_review.head(9))

## 8. Results and takeaways

Run this section after metrics and manual scores are complete. Conclusions must distinguish preservation, semantic edit success, and overall visual quality. The fixed oracle gate and dynamic part-to-edit transfer should be interpreted separately.

In [ ]:
manual_scored = pd.read_csv(MANUAL_PATH)
score_names = [
    "local_edit_success_0_2",
    "non_target_preservation_0_2",
    "overall_visual_quality_0_2",
]
for column in score_names:
    manual_scored[column] = pd.to_numeric(manual_scored[column], errors="coerce")

manual_summary = manual_scored.groupby("method")[score_names].agg(["count", "mean"])
display(manual_summary.round(3))

if manual_scored[score_names].notna().to_numpy().all():
    display(Markdown(
        "### Interpretation checklist\n"
        "- Compare local-edit success separately from preservation.\n"
        "- Treat identical seed outputs as one unique output.\n"
        "- Inspect cases where preservation improves while semantic success falls.\n"
        "- Confirm that metric differences are visually meaningful in the full comparison."
    ))
else:
    missing_scores = int(manual_scored[score_names].isna().sum().sum())
    display(Markdown(
        f"**Manual assessment is incomplete:** {missing_scores} score cells remain empty. "
        "Do not finalize semantic conclusions until they are filled."
    ))